# 增量语义 Gaussian 地图 · Colab 实验入口

先选择 **Runtime → Change runtime type → GPU**。此 notebook 在当前 Mac 上仅做了结构/语法检查，GPU 推理与 CUDA 编译尚未执行验证。按顺序运行，任何 smoke test 失败时先修复环境，再运行数据实验。

输入位姿来自真值，因此本版本研究建图，不是完整 SLAM。合成序列的分割来自真值；TUM 分割来自 Mask2Former。分割先缓存、随后运行建图，两者耗时分别报告。

上传随附的 `semantic-gaussian-lab.zip`。真实 TUM 数据下载约 0.42 GB，模型权重另行下载。初始真实实验仅 40 帧。

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess, json
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith('.zip')), None)
assert archive, '请上传 semantic-gaussian-lab.zip'
destination = Path('/content/sglab-project')
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        assert (destination/item.filename).resolve().is_relative_to(destination.resolve()), 'invalid zip path'
    z.extractall(destination)
project = next(destination.rglob('pyproject.toml')).parent
os.chdir(project)
print('project:', project)

In [ ]:
%pip install -q -e '.[segmentation]' 'gsplat>=1.5,<2' ninja
import torch, shutil, platform
assert torch.cuda.is_available(), '请选择 GPU runtime 后重新运行'
environment = {'python': platform.python_version(), 'torch':torch.__version__,
               'cuda':torch.version.cuda, 'gpu':torch.cuda.get_device_name(), 'nvcc':shutil.which('nvcc')}
Path('results').mkdir(exist_ok=True)
Path('results/colab-environment.json').write_text(json.dumps(environment,indent=2))
Path('results/pip-freeze.txt').write_bytes(subprocess.check_output([sys.executable,'-m','pip','freeze']))
print(environment)

## 先验证最小计算链路

第一条测试不依赖模型权重；第二条触发 gsplat CUDA 构建并检查 RGB+depth、134 通道语义渲染和反向梯度。首次 JIT 编译可能耗时数分钟。第三条才跑 12 帧合成数据的局部优化。

In [ ]:
def run(*arguments):
    subprocess.run([sys.executable,*arguments],check=True)
run('-m','unittest','discover','-s','tests','-v')
run('-m','sglab.gpu_smoke')
run('-m','sglab','run','--frames','12','--max-gaussians','8000','--refine','--output','results/gpu-smoke')

## 下载和转换真实 RGB-D

TUM xyz 适合检查尺度和数据接入，不能代表整个房间的长距离扩展，也没有完整三维语义/实例真值。默认 ROS 内参、深度/5000、RGB-depth-pose 最近时间关联 ≤25 ms，缩小至 320×240。官方格式说明：[TUM](https://cvg.cit.tum.de/data/datasets/rgbd-dataset/file_formats)。后续可换为自己准备的通用 manifest。

In [ ]:
import urllib.request, tarfile
RUN_REAL_DATA = True
if RUN_REAL_DATA:
    raw = Path('/content/tum-raw'); raw.mkdir(exist_ok=True)
    archive_path = raw/'tum-xyz.tgz'
    url = 'https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_xyz.tgz'
    if not archive_path.exists():
        print('Downloading TUM xyz (~0.42 GB)...')
        urllib.request.urlretrieve(url,archive_path)
    with tarfile.open(archive_path) as archive_file:
        archive_file.extractall(raw,filter='data')
    dataset = raw/'rgbd_dataset_freiburg1_xyz'
    run('-m','sglab','import-tum','--dataset',str(dataset),'--output','data/tum',
        '--frames','40','--every','6')
    manifest = 'data/tum/manifest.json'
else:
    run('-m','sglab','make-demo','--frames','40','--output','data/synthetic')
    manifest = 'data/synthetic/manifest.json'
print(manifest)

## 自动语义与实例分割

Mask2Former COCO panoptic 同时预测 thing / stuff。unknown 保留为 0；二维实例 ID 每帧独立，地图稍后通过三维关联建立全局 ID。COCO 类别词表与 ScanNet/Replica 不是同一套，正式评估需做 label mapping。这里每帧都推理以简化正确性；分割不会被算入后续在线地图融合耗时。

In [ ]:
if RUN_REAL_DATA:
    panoptic = 'data/tum/panoptic.json'
    run('-m','sglab','segment','--manifest',manifest,'--output',panoptic,'--device','cuda')
else:
    panoptic = manifest  # synthetic oracle segmentation, NOT learned predictions
print(panoptic)

## 增量融合和 CUDA 局部优化

最大 24,000 Gaussians；CPU 融合目标 45 ms 是软预算，不包含 CUDA 优化或分割。CUDA 每 5 帧优化 5 步，至多 6,000 活动点；固定旧稳定区与早期关键帧回放。必须同时查看 CPU、refine 和总耗时，不能将 CPU target 当成端到端实时保证。

In [ ]:
run('-m','sglab','run','--manifest',panoptic,'--frames','40','--max-gaussians','24000',
    '--target-ms','45','--refine','--output','results/real-gpu' if RUN_REAL_DATA else 'results/synthetic-gpu')
result = Path('results/real-gpu' if RUN_REAL_DATA else 'results/synthetic-gpu')
summary = json.loads((result/'summary.json').read_text())
summary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
metrics = json.loads((result/'metrics.json').read_text())
frame_ids = [m['frame'] for m in metrics]
fig, axes = plt.subplots(1,3,figsize=(15,3.5))
axes[0].plot(frame_ids,[m['gaussians'] for m in metrics]); axes[0].axhline(24000,color='gray',linestyle='--')
axes[0].set(xlabel='Frame',ylabel='Gaussian count')
axes[1].plot(frame_ids,[m['update_ms'] for m in metrics],label='CPU fusion')
axes[1].plot(frame_ids,[m['mapping_total_ms'] for m in metrics],label='Fusion + refinement')
axes[1].set(xlabel='Frame',ylabel='Time (ms)'); axes[1].legend()
gpu = [(m['frame'],m['gpu_peak_mib']) for m in metrics if 'gpu_peak_mib' in m]
if gpu: axes[2].plot(*zip(*gpu))
axes[2].set(xlabel='Frame',ylabel='Measured CUDA peak (MiB)')
fig.tight_layout(); fig.savefig(result/'performance.png',dpi=160)
plt.show()
cached = [f.get('segmentation_ms') for f in json.loads(Path(panoptic).read_text())['frames']]
cached = [v for v in cached if v is not None]
if cached: print('Separate cached segmentation p50/p95 (ms):',np.percentile(cached,[50,95]))
print('Probe quality values are centre-projection diagnostics, NOT full rendered-view PSNR.')

## 可选：同输入的 CPU 预算对照

以下是管线不变量和性能的合成对照，不是训练质量结果。每组独立进程测峰值 RSS，固定采样量和随机种子。正式研究还要对真实场景做多次重复、完整渲染质量和三维语义实例指标。

In [ ]:
RUN_CPU_ABLATIONS = False
if RUN_CPU_ABLATIONS:
    run('-m','sglab.experiments','--output','results/ablations','--frames','100')

## 下载结果与真实输入

`map.ply` 包含 Gaussian 属性和语义/实例 ID；`map.npz` 保留证据与元数据。下载的 `data/` 可在本地查看器重放并重新执行 CPU 融合，不会自动应用 CUDA 优化地图。要检查优化结果请打开 PLY。保存 notebook 输出以保留安装/编译错误的证据。

In [ ]:
export_file = Path('/content/semantic-gaussian-results.zip')
with zipfile.ZipFile(export_file,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [Path('results'),Path('data')]:
        for file in folder.rglob('*'):
            if file.is_file(): z.write(file,file.as_posix())
files.download(str(export_file))